In [2]:
# Install first:
# pip install -U langchain langchain-core langchain-community langchain-groq langchain-huggingface chromadb sentence-transformers torch

# Optional: set Groq API key inside notebook
# import os
# os.environ["GROQ_API_KEY"] = "your_groq_api_key"

# Imports Document class from updated LangChain package
from langchain_core.documents import Document

# Imports Chroma vector database
from langchain_community.vectorstores import Chroma

# Imports open-source Hugging Face embedding support
from langchain_huggingface import HuggingFaceEmbeddings

# Imports Groq chat model integration
from langchain_groq import ChatGroq

# Imports prompt template helper
from langchain_core.prompts import ChatPromptTemplate

# Imports parser to convert LLM output into plain string
from langchain_core.output_parsers import StrOutputParser

# Create an open-source embedding model instead of OpenAIEmbeddings
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Create documents for the knowledge base
docs = [
    Document(
        page_content="Peak Performance Gym was founded in 2015 by former Olympic athlete Marcus Chen. With over 15 years of experience in professional athletics, Marcus established the gym to provide personalized fitness solutions for people of all levels. The gym spans 10,000 square feet and features state-of-the-art equipment.",
        metadata={"source": "about.txt"}
    ),
    Document(
        page_content="Peak Performance Gym is open Monday through Friday from 5:00 AM to 11:00 PM. On weekends, our hours are 7:00 AM to 9:00 PM. We remain closed on major national holidays. Members with Premium access can enter using their key cards 24/7, including holidays.",
        metadata={"source": "hours.txt"}
    ),
    Document(
        page_content="Our membership plans include: Basic (₹1,500/month) with access to gym floor and basic equipment; Standard (₹2,500/month) adds group classes and locker facilities; Premium (₹4,000/month) includes 24/7 access, personal training sessions, and spa facilities. We offer student and senior citizen discounts of 15% on all plans. Corporate partnerships are available for companies with 10+ employees joining.",
        metadata={"source": "membership.txt"}
    ),
    Document(
        page_content="Group fitness classes at Peak Performance Gym include Yoga (beginner, intermediate, advanced), HIIT, Zumba, Spin Cycling, CrossFit, and Pilates. Beginner classes are held every Monday and Wednesday at 6:00 PM. Intermediate and advanced classes are scheduled throughout the week. The full schedule is available on our mobile app or at the reception desk.",
        metadata={"source": "classes.txt"}
    ),
    Document(
        page_content="Personal trainers at Peak Performance Gym are all certified professionals with minimum 5 years of experience. Each new member receives a complimentary fitness assessment and one free session with a trainer. Our head trainer, Neha Kapoor, specializes in rehabilitation fitness and sports-specific training. Personal training sessions can be booked individually (₹800/session) or in packages of 10 (₹7,000) or 20 (₹13,000).",
        metadata={"source": "trainers.txt"}
    ),
    Document(
        page_content="Peak Performance Gym's facilities include a cardio zone with 30+ machines, strength training area, functional fitness space, dedicated yoga studio, spin class room, swimming pool (25m), sauna and steam rooms, juice bar, and locker rooms with shower facilities. Our equipment is replaced or upgraded every 3 years to ensure members have access to the latest fitness technology.",
        metadata={"source": "facilities.txt"}
    )
]

# Create a Chroma vector database from documents using open-source embeddings
db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_function
)

# Create a retriever using MMR search
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

# Test retrieval
retrieved_docs = retriever.invoke("Who founded the gym and what are the timings?")

# Print retrieved documents for checking
for doc in retrieved_docs:
    print(doc.metadata)
    print(doc.page_content)
    print("-" * 80)

# Create a prompt template for answering only from retrieved context
template = """
Answer the question based only on the following context:

{context}

Question: {question}

If the answer is not available in the context, say:
"I don't know based on the provided context."
"""

# Convert the template string into a LangChain prompt object
prompt = ChatPromptTemplate.from_template(template)

# Create Groq LLM instead of ChatOpenAI
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Function to combine retrieved documents into one context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the full RAG chain
qa_chain = (
    {
        "context": lambda question: format_docs(retriever.invoke(question)),
        "question": lambda question: question
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Ask a question
answer = qa_chain.invoke("Who is the owner and what are the timings?")

# Print the final answer
print(answer)

c:\Users\HP\Desktop\Jupyter Notebooks\Projects\LangGraph\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\HP\Desktop\Jupyter Notebooks\Projects\LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\HP\Desktop\Jupyter Notebooks\Projects\LangGraph\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your dis

{'source': 'about.txt'}
Peak Performance Gym was founded in 2015 by former Olympic athlete Marcus Chen. With over 15 years of experience in professional athletics, Marcus established the gym to provide personalized fitness solutions for people of all levels. The gym spans 10,000 square feet and features state-of-the-art equipment.
--------------------------------------------------------------------------------
{'source': 'classes.txt'}
Group fitness classes at Peak Performance Gym include Yoga (beginner, intermediate, advanced), HIIT, Zumba, Spin Cycling, CrossFit, and Pilates. Beginner classes are held every Monday and Wednesday at 6:00 PM. Intermediate and advanced classes are scheduled throughout the week. The full schedule is available on our mobile app or at the reception desk.
--------------------------------------------------------------------------------
{'source': 'membership.txt'}
Our membership plans include: Basic (₹1,500/month) with access to gym floor and basic equipment;